# 30-day sales forecasting: Baseline vs LightGBM vs ModernTCN vs TiDE

Forecast **daily sales for each of the 1,235 products, 30 days ahead**, and compare four methods on exactly the same data, inputs and test windows:

| # | Method | Type |
|---|--------|------|
| 1 | **Baseline: same weekday last week** | repeat the last observed week; every model has to beat this |
| 2 | **LightGBM** | one global gradient-boosted tree model, direct multi-horizon |
| 3 | **ModernTCN** | official architecture from `Mr0022/ModernTCN` (large-kernel temporal conv net) |
| 4 | **TiDE** | dense encoder–decoder for covariates (Das et al., 2023) |

**Shared setup**
- **Input window:** the last **112 days** (16 weeks) before the forecast origin. **Output:** the next **30 days**.
- **Inputs:**
  - sales history;
  - discount, store availability, relative price (past values only);
  - calendar (Jalali dates, Nowruz, Tasua/Ashura; known in advance);
  - product category (brand, Level1–5).

  Each model uses them in its native way. ModernTCN's architecture has no slot for calendar or category inputs, so it only gets the four past series.
- **Rolling-origin test:** 3 back-to-back 30-day windows covering the last 90 days. Models are trained once on data **before** the validation window, early-stopped on validation, then forecast each test window from the history available at that origin. No future information is used.
- **Scored against actual sales** (`sale_qty_raw`, before outlier capping). Days outside a product's lifespan are excluded.

**Run on Colab (GPU):** set *Runtime → Change runtime type → GPU* and run all cells. The setup cell clones the repositories and builds the cleaned data if needed. Set `SMOKE_TEST = True` for a quick one-epoch check.

## 1. Setup

In [ ]:
import os, sys, time, subprocess, warnings
from pathlib import Path
from types import SimpleNamespace

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0") == "1"   # True -> 1 epoch per deep model, quick consistency check

IN_COLAB = "google.colab" in sys.modules
OFOGH_REPO = "https://github.com/Mr0022/ofogh-kurush.git"
OFOGH_BRANCH = "claude/gallant-gauss-5hdb4c"
MODERNTCN_REPO = "https://github.com/Mr0022/ModernTCN.git"

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if IN_COLAB:
    if not Path("/content/ofogh-kurush").exists():
        sh(f"git clone -q -b {OFOGH_BRANCH} {OFOGH_REPO} /content/ofogh-kurush")   # private repo? upload the folder instead
    os.chdir("/content/ofogh-kurush/notebooks")
    sh("pip -q install lightgbm")

# repository root = folder that holds candidate_dataset.parquet
REPO_ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "candidate_dataset.parquet").exists())
OUT_DIR = REPO_ROOT / "outputs"
PANEL_PATH = OUT_DIR / "panel_clean.parquet"

# cleaned panel comes from notebooks/eda_and_preprocessing.ipynb; build it if missing
if not PANEL_PATH.exists():
    print("panel_clean.parquet not found -> running the EDA/preprocessing notebook first (~1 min)")
    sh(f"jupyter nbconvert --to notebook --execute '{REPO_ROOT / 'notebooks/eda_and_preprocessing.ipynb'}' "
       f"--output '{OUT_DIR.parent / 'notebooks/_eda_executed.ipynb'}' --ExecutePreprocessor.timeout=1800")

# official ModernTCN code
MODERNTCN_DIR = next((p for p in [Path(os.environ.get("MODERNTCN_DIR", "/nonexistent")), REPO_ROOT.parent / "ModernTCN",
                                  REPO_ROOT.parent / "mr0022/moderntcn", Path("/content/ModernTCN")]
                      if (p / "ModernTCN-Long-term-forecasting").exists()), None)
if MODERNTCN_DIR is None:
    MODERNTCN_DIR = REPO_ROOT.parent / "ModernTCN"
    sh(f"git clone -q --depth 1 {MODERNTCN_REPO} '{MODERNTCN_DIR}'")
sys.path.insert(0, str(MODERNTCN_DIR / "ModernTCN-Long-term-forecasting"))
print("repo:", REPO_ROOT, "| ModernTCN:", MODERNTCN_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import lightgbm as lgb
import torch
import torch.nn as nn
from models.ModernTCN import Model as ModernTCNModel      # from the cloned ModernTCN repo

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

# ---- forecasting setup ----
L, H = 112, 30                       # look-back window, horizon (days)
N_TEST_FOLDS, N_VAL_DAYS = 3, 60     # 3 x 30-day test windows; 60-day validation block before them

# ---- deep-learning training budget ----
EPOCHS            = 1 if SMOKE_TEST else 40
SAMPLES_PER_EPOCH = 20_000 if SMOKE_TEST else 200_000
BATCH_SIZE        = 512
PATIENCE          = 6
DL_VOLUME_WEIGHTS = False            # weight the NN loss by product volume (tested: slightly worse for both NNs)

# ---- LightGBM ----
LGB_ORIGIN_STRIDE = 6                # training origins every 6 days (cycles through all weekdays)
LGB_ROUNDS        = 3000
LGB_PARAMS = dict(objective="l1",                 # L1 in log space = median forecast (what WAPE rewards)
                   learning_rate=0.05, num_leaves=127, min_data_in_leaf=200,
                  feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  max_cat_to_onehot=8, cat_smooth=20, verbose=-1, seed=SEED, num_threads=os.cpu_count())

# ---- plotting style ----
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, INK2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlecolor": INK, "axes.titlesize": 12,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "axes.prop_cycle": plt.cycler(color=PALETTE), "lines.linewidth": 1.5, "legend.frameon": False,
    "figure.dpi": 100, "font.size": 10, "figure.figsize": (12, 4),
})
MODEL_COLORS = {"Baseline (same weekday last week)": MUTED, "LightGBM": PALETTE[0], "ModernTCN": PALETTE[1], "TiDE": PALETTE[2]}

## 2. Data
The cleaned long panel comes from `eda_and_preprocessing.ipynb`: missing rows imputed, corrupted dates repaired, price outliers fixed, unexplained sales spikes capped. Here it is reshaped into aligned `[days × products]` matrices.

In [ ]:
panel = pd.read_parquet(PANEL_PATH)
products = sorted(panel.product_id.unique())
cal = panel[["ds", "jdate"]].drop_duplicates().sort_values("ds").reset_index(drop=True)
dates = pd.DatetimeIndex(cal.ds)
T, N = len(dates), len(products)

def wide(col):
    return panel.pivot(index="ds", columns="product_id", values=col).reindex(index=dates, columns=products)

sales_df = wide("sale_qty")
INSIDE = sales_df.notna().values                                   # inside product lifespan
SALES = sales_df.fillna(0.0).values.astype("float32")              # cleaned (model input & training target)
SALES_RAW = wide("sale_qty_raw").fillna(0.0).values.astype("float32")   # actual sales (evaluation truth)
Y = np.log1p(SALES)                                                # modelling scale

# ---- split (indices of the first forecast day = "origin") ----
TEST_ORIGINS = [T - H * k for k in range(N_TEST_FOLDS, 0, -1)]            # e.g. T-90, T-60, T-30
VAL_START = TEST_ORIGINS[0] - N_VAL_DAYS
VAL_ORIGINS = list(range(VAL_START, TEST_ORIGINS[0] - H + 1, 7))          # targets stay inside the validation block
TRAIN_LAST_ORIGIN = VAL_START - H                                          # training targets end before validation
TRAIN_ORIGINS = np.arange(L, TRAIN_LAST_ORIGIN + 1)

# ---- covariates (past only); scalers use the training period only ----
disc = (wide("discount_pct").ffill().fillna(0.0) / 100).values
stores = wide("num_stores_with_stock").ffill().fillna(0.0)
avail = (stores / stores.iloc[:VAL_START].quantile(0.95).replace(0, 1)).clip(0, 1).values
price = wide("price").ffill().bfill()
price_rel = np.log(price / price.rolling(28, min_periods=1).median()).fillna(0.0).values     # trailing window
COV = np.stack([disc, avail, price_rel], axis=-1).astype("float32")                          # [T, N, 3]
COV_NAMES = ["discount", "availability", "price_rel"]

# ---- static categories ----
meta = panel.groupby("product_id")[["brand_id", "Level1", "Level2", "Level3", "Level4", "Level5"]].first().loc[products]
STATIC = np.stack([meta[c].astype("category").cat.codes.values for c in meta.columns], axis=1).astype("int64")   # [N, 6]
CAT_SIZES = [int(meta[c].nunique()) for c in meta.columns]

print(f"{N} products x {T} days ({dates[0].date()} -> {dates[-1].date()})")
print(f"train origins : {dates[TRAIN_ORIGINS[0]].date()} .. {dates[TRAIN_ORIGINS[-1]].date()} "
      f"(targets until {dates[TRAIN_LAST_ORIGIN + H - 1].date()})")
print(f"val origins   : {[str(dates[o].date()) for o in VAL_ORIGINS]}")
print(f"test windows  : {[f'{dates[o].date()} -> {dates[o + H - 1].date()}' for o in TEST_ORIGINS]}")

In [ ]:
# ---- calendar (Jalali + Iranian holidays): known in advance for any horizon ----
j_year = cal.jdate.str[:4].astype(int).values
j_month = cal.jdate.str[4:6].astype(int).values
j_day = cal.jdate.str[6:8].astype(int).values
j_doy = np.where(j_month <= 6, (j_month - 1) * 31, 186 + (j_month - 7) * 30) + j_day
dow = (dates.day_of_week.values + 2) % 7                                  # Saturday = 0 (Iranian week)

# next Nowruz (1 Farvardin): the first day with j_month == 1 & j_day == 1 at or after each date
nowruz_idx = np.where((j_month == 1) & (j_day == 1))[0]
next_nowruz = np.array([next((n for n in nowruz_idx if n >= t), t + 400) for t in range(T)])
days_to_nowruz = np.clip(next_nowruz - np.arange(T), 0, 30)

TASUA_ASHURA = pd.to_datetime(["2025-07-05", "2025-07-06", "2026-06-24", "2026-06-25"])   # extend for future horizons
OTHER_SOLAR = {(1, 12), (3, 14), (3, 15), (11, 22), (12, 29)}
CAL = pd.DataFrame({
    "dow": dow, "j_month": j_month, "j_day": j_day, "j_doy": j_doy, "days_to_nowruz": days_to_nowruz,
    "is_nowruz": ((j_month == 1) & (j_day <= 4)).astype(int),
    "is_sizdah": ((j_month == 1) & (j_day == 13)).astype(int),
    "is_tasua_ashura": dates.isin(TASUA_ASHURA).astype(int),
    "is_other_holiday": np.array([(m, d) in OTHER_SOLAR for m, d in zip(j_month, j_day)]).astype(int),
}, index=dates)

# scaled version for neural nets ([-0.5, 0.5] like Time-Series-Library timeenc=1, flags in {0, 1})
MARKS = np.column_stack([
    dow / 6 - 0.5, (j_day - 1) / 30 - 0.5, (j_month - 1) / 11 - 0.5, (j_doy - 1) / 365 - 0.5, days_to_nowruz / 30 - 0.5,
    np.sin(2 * np.pi * dow / 7), np.cos(2 * np.pi * dow / 7),
    CAL.is_nowruz, CAL.is_sizdah, CAL.is_tasua_ashura, CAL.is_other_holiday,
]).astype("float32")
print("calendar features:", list(CAL.columns), "| NN marks:", MARKS.shape)
print("test-period holidays:", [str(d.date()) for d in dates[TEST_ORIGINS[0]:] if CAL.loc[d, ["is_tasua_ashura", "is_other_holiday", "is_nowruz", "is_sizdah"]].any()])

In [ ]:
total = SALES_RAW.sum(axis=1)
fig, ax = plt.subplots(figsize=(14, 3.6))
ax.plot(dates, total / 1e6, color=MUTED, lw=0.8)
spans = [("train targets", L, TRAIN_LAST_ORIGIN + H, PALETTE[0]), ("validation", VAL_START, TEST_ORIGINS[0], PALETTE[3]),
         ("test (3 x 30 days)", TEST_ORIGINS[0], T, PALETTE[1])]
for name, a, b, c in spans:
    ax.axvspan(dates[a], dates[b - 1], color=c, alpha=0.12, lw=0, label=name)
for o in TEST_ORIGINS:
    ax.axvline(dates[o], color=PALETTE[1], lw=1)
ax.set_title("Total daily sales (millions) with the rolling-origin split"); ax.legend(loc="upper left", ncol=3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.show()

## 3. Evaluation
For every test window (origin `o`, days `o … o+29`) and product:
- **WAPE** = Σ|forecast − actual| / Σ actual. This is the main metric: volume-weighted, and readable as "% of units mis-forecast".
- **Bias** = Σ(forecast − actual) / Σ actual. Positive means over-forecasting.
- **RMSE** and **MAE** in units.
- **Skill vs baseline** = 1 − WAPE_model / WAPE_baseline. Above 0 means the model beats the baseline.
- **Product win rate** = share of products where the model's WAPE is lower than the baseline's.

In [ ]:
FORECASTS = {}   # model name -> array [n_folds, H, N] in units

def evaluate(pred):
    pred = np.asarray(pred, dtype="float64")
    truth = np.stack([SALES_RAW[o:o + H] for o in TEST_ORIGINS]).astype("float64")
    mask = np.stack([INSIDE[o:o + H] for o in TEST_ORIGINS])
    err = np.where(mask, pred - truth, 0.0)
    act = np.where(mask, truth, 0.0)
    return {
        "WAPE": np.abs(err).sum() / act.sum(),
        "Bias": err.sum() / act.sum(),
        "MAE": np.abs(err).sum() / mask.sum(),
        "RMSE": np.sqrt((err ** 2).sum() / mask.sum()),
        "fold_WAPE": np.abs(err).sum(axis=(1, 2)) / act.sum(axis=(1, 2)),
        "horizon_WAPE": np.abs(err).sum(axis=(0, 2)) / act.sum(axis=(0, 2)),
        "product_WAPE": np.abs(err).sum(axis=(0, 1)) / np.maximum(act.sum(axis=(0, 1)), 1e-9),
        "total_WAPE": np.abs(err.sum(axis=2)).sum() / act.sum(),   # error of the all-product daily total
    }

def summary_table():
    base = evaluate(FORECASTS["Baseline (same weekday last week)"])
    rows = {}
    for name, p in FORECASTS.items():
        m = evaluate(p)
        rows[name] = {"WAPE": m["WAPE"], "skill vs baseline": 1 - m["WAPE"] / base["WAPE"], "Bias": m["Bias"],
                      "MAE": m["MAE"], "RMSE": m["RMSE"], "total-sales WAPE": m["total_WAPE"],
                      "product win rate": np.mean(m["product_WAPE"] < base["product_WAPE"]),
                      **{f"WAPE fold {k + 1}": v for k, v in enumerate(m["fold_WAPE"])}}
    return pd.DataFrame(rows).T

def predict_to_units(log_pred):
    return np.clip(np.expm1(log_pred), 0, None)

## 4. Baseline: same weekday last week
The forecast for any day in the next 30 is the value from the **same weekday in the last observed week**, i.e. the last 7 days are repeated.

In [ ]:
def seasonal_naive(o):
    return SALES[o - 7:o][np.arange(H) % 7]          # day o+h  <-  day o-7+(h mod 7)

FORECASTS["Baseline (same weekday last week)"] = np.stack([seasonal_naive(o) for o in TEST_ORIGINS])
summary_table().round(3)

## 5. LightGBM: global direct multi-horizon model
One model for all products and all 30 horizons. Each training row is *(product, origin, horizon h)*. Its features describe the history **up to the origin** plus the **calendar of the target day**:

| Group | Features |
|---|---|
| Level | `level` = mean log-sales over the last 28 days (all other sales features are relative to it, so the model is scale-free) |
| Recent sales | lags 1–7, rolling means 7/14/56/112, std 28, zero-share 28 |
| Weekly pattern | same weekday as the target day, 1–4 weeks back (and their mean) |
| Covariates at origin | discount (last, 7d, 28d mean), availability (last, 7d, 28d), relative price (last, 7d), stock-out share 28d |
| Market | momentum of total market sales (7d vs 28d) |
| Target day | horizon `h`, day-of-week, Jalali month/day/day-of-year, days to Nowruz, holiday flags |
| Static | brand, Level1–Level5 (categorical) |

**Target:** `log1p(sales[o+h]) − level`. The forecast is `expm1(prediction + level)`.

**Loss:** L1 in log space, which gives a median forecast (what WAPE rewards). Rows are **weighted by product volume**, so big sellers count as much as they do in WAPE. In tests this was clearly better than an unweighted L2 loss (WAPE 26.9% vs 29.6%). Early stopping on the validation windows is reliable here: test error also rises when the model is trained for more rounds.

In [ ]:
def csum(A):
    return np.concatenate([np.zeros((1,) + A.shape[1:]), np.cumsum(A, axis=0, dtype="float64")])

CS = {"y": csum(Y), "y2": csum(Y.astype("float64") ** 2), "zero": csum((SALES == 0).astype("float32")),
      "disc": csum(COV[..., 0]), "avail": csum(COV[..., 1]), "prel": csum(COV[..., 2]),
      "stockout": csum((stores.values < 0.5).astype("float32")), "inside": csum(INSIDE.astype("float32"))}
MKT = np.log1p(SALES.sum(axis=1)); CS["mkt"] = csum(MKT)

def rmean(key, o, k):                      # mean over days [o-k, o) -> [n_o, N] (or [n_o] for market)
    return ((CS[key][o] - CS[key][o - k]) / k).astype("float32")

STATIC_COLS = list(meta.columns)

def lgb_frame(origins, require_full_window=True):
    o = np.asarray(origins)
    h = np.arange(H)
    n_o = len(o)
    level = rmean("y", o, 28)                                              # [n_o, N]
    f = {}
    for k in range(1, 8):
        f[f"lag{k}"] = Y[o - k] - level
    for k in (7, 14, 56, 112):
        f[f"mean{k}"] = rmean("y", o, k) - level
    f["std28"] = np.sqrt(np.maximum(rmean("y2", o, 28) - level.astype("float64") ** 2, 0)).astype("float32")
    f["zero28"] = rmean("zero", o, 28)
    f["disc_last"], f["disc_m7"], f["disc_m28"] = COV[o - 1, :, 0], rmean("disc", o, 7), rmean("disc", o, 28)
    f["avail_last"], f["avail_m7"], f["avail_m28"] = COV[o - 1, :, 1], rmean("avail", o, 7), rmean("avail", o, 28)
    f["prel_last"], f["prel_m7"] = COV[o - 1, :, 2], rmean("prel", o, 7)
    f["stockout28"] = rmean("stockout", o, 28)
    f["level"] = level
    f["mkt_mom"] = np.repeat((rmean("mkt", o, 7) - rmean("mkt", o, 28))[:, None], N, axis=1)

    # broadcast origin features over the horizon: [n_o, H, N]
    feats = {k: np.broadcast_to(v[:, None, :], (n_o, H, N)) for k, v in f.items()}
    # weekday-aligned lags: same weekday as the target day, 1..4 weeks back
    sw = [Y[o[:, None] - 7 * k + (h % 7)[None, :]] - level[:, None, :] for k in range(1, 5)]
    for k, s in enumerate(sw, 1):
        feats[f"sw{k}"] = s
    feats["sw_mean"] = np.mean(sw, axis=0)
    tgt_idx = o[:, None] + h[None, :]                                      # [n_o, H]
    feats["h"] = np.broadcast_to(h[None, :, None], (n_o, H, N))
    for c in CAL.columns:
        feats[c] = np.broadcast_to(CAL[c].values[tgt_idx][:, :, None], (n_o, H, N))
    for j, c in enumerate(STATIC_COLS):
        feats[c] = np.broadcast_to(STATIC[None, None, :, j], (n_o, H, N))

    X = pd.DataFrame({k: np.ascontiguousarray(v).reshape(-1) for k, v in feats.items()})
    for c in X.columns:
        if X[c].dtype == "float64": X[c] = X[c].astype("float32")
    target = (Y[tgt_idx] - level[:, None, :]).reshape(-1)
    level_flat = np.broadcast_to(level[:, None, :], (n_o, H, N)).reshape(-1)
    if require_full_window:                                                # whole window inside the product's life
        valid = ((CS["inside"][o + H] - CS["inside"][o - L]) == L + H)
        keep = np.broadcast_to(valid[:, None, :], (n_o, H, N)).reshape(-1)
    else:
        keep = np.ones(len(X), dtype=bool)
    return X[keep].reset_index(drop=True), target[keep], level_flat[keep]

t0 = time.time()
train_orig = TRAIN_ORIGINS[::LGB_ORIGIN_STRIDE]
X_tr, y_tr, _ = lgb_frame(train_orig)
X_va, y_va, _ = lgb_frame(VAL_ORIGINS)
print(f"LightGBM rows: train {len(X_tr):,} ({len(train_orig)} origins) | val {len(X_va):,} | {X_tr.shape[1]} features "
      f"| built in {time.time() - t0:.0f}s")

In [ ]:
t0 = time.time()
# weight rows by product volume (units/day over the last 28 days) so training matches the volume-weighted WAPE
vol_w = lambda X: np.expm1(X["level"].values).clip(1, None)
dtrain = lgb.Dataset(X_tr, y_tr, weight=vol_w(X_tr), categorical_feature=STATIC_COLS, free_raw_data=False)
dval = lgb.Dataset(X_va, y_va, weight=vol_w(X_va), categorical_feature=STATIC_COLS, reference=dtrain)
lgb_model = lgb.train(LGB_PARAMS, dtrain, num_boost_round=LGB_ROUNDS, valid_sets=[dval], valid_names=["val"],
                      callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(250)])
print(f"best iteration {lgb_model.best_iteration} | val weighted L1 {lgb_model.best_score['val']['l1']:.4f} | {time.time() - t0:.0f}s")

def lgb_forecast(o):
    X, _, level = lgb_frame([o], require_full_window=False)
    p = lgb_model.predict(X, num_iteration=lgb_model.best_iteration) + level
    return predict_to_units(p).reshape(H, N)

FORECASTS["LightGBM"] = np.stack([lgb_forecast(o) for o in TEST_ORIGINS])

imp = pd.Series(lgb_model.feature_importance("gain"), index=X_tr.columns).sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.barh(imp.index[::-1], imp.values[::-1] / imp.sum(), color=PALETTE[0], height=0.6)
ax.set_title("LightGBM: top 15 features by gain (share of top 15)")
plt.tight_layout(); plt.show()
summary_table().round(3)

## 6. Deep-learning models: shared data pipeline & training loop
Both neural models are **global**: one network for all products. A sample is *(product, origin)* with:
- `y_hist` [112]: log1p sales;
- `past_cov` [112 × 3]: discount, availability, relative price;
- `marks` [142 × 11]: calendar for history + horizon;
- `static` [6]: category ids;
- target `y_fut` [30].

Training uses random samples from all training origins, an **L1 loss in log space** (a median forecast, which suits WAPE), AdamW, and early stopping on the validation windows. Both models get exactly the same training budget.

In [ ]:
Yt = torch.tensor(Y, device=DEVICE)
COVt = torch.tensor(COV, device=DEVICE)
MARKSt = torch.tensor(MARKS, device=DEVICE)
STATICt = torch.tensor(STATIC, device=DEVICE)
AR_L = torch.arange(-L, 0, device=DEVICE)
AR_H = torch.arange(0, H, device=DEVICE)

def make_pairs(origins, require_full_window=True):
    o = np.asarray(origins)
    valid = ((CS["inside"][o + H] - CS["inside"][o - L]) == L + H) if require_full_window else np.ones((len(o), N), bool)
    oi, pi = np.nonzero(valid)
    return np.stack([pi, o[oi]], axis=1)                                   # [n, 2] = (product, origin)

TRAIN_PAIRS, VAL_PAIRS = make_pairs(TRAIN_ORIGINS), make_pairs(VAL_ORIGINS)
print(f"training samples: {len(TRAIN_PAIRS):,} | validation samples: {len(VAL_PAIRS):,}")

def get_batch(pairs):
    p = torch.as_tensor(pairs[:, 0], device=DEVICE)
    o = torch.as_tensor(pairs[:, 1], device=DEVICE)
    t_hist, t_fut = o[:, None] + AR_L[None, :], o[:, None] + AR_H[None, :]
    return dict(y_hist=Yt[t_hist, p[:, None]], past_cov=COVt[t_hist, p[:, None]],
                marks=MARKSt[torch.cat([t_hist, t_fut], dim=1)], static=STATICt[p], y_fut=Yt[t_fut, p[:, None]])

def fit_dl(name, model, lr):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(EPOCHS, 1))
    rng = np.random.default_rng(SEED)
    best, best_state, wait, hist = np.inf, None, 0, []
    print(f"{name}: {sum(p.numel() for p in model.parameters()):,} parameters")
    for ep in range(EPOCHS):
        t0 = time.time(); model.train(); tr_loss = []
        idx = rng.choice(len(TRAIN_PAIRS), size=min(SAMPLES_PER_EPOCH, len(TRAIN_PAIRS)), replace=False)
        for b in range(0, len(idx), BATCH_SIZE):
            batch = get_batch(TRAIN_PAIRS[idx[b:b + BATCH_SIZE]])
            err = (model(batch) - batch["y_fut"]).abs()
            if DL_VOLUME_WEIGHTS:
                w = torch.expm1(batch["y_hist"][:, -28:].mean(1, keepdim=True)).clamp(min=1)
                loss = (err * w).sum() / (w.sum() * H)
            else:
                loss = err.mean()
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            tr_loss.append(loss.item())
        sched.step()
        va_loss = dl_loss(model, VAL_PAIRS)
        hist.append((np.mean(tr_loss), va_loss))
        print(f"  epoch {ep + 1:2d}  train L1 {np.mean(tr_loss):.4f}  val L1 {va_loss:.4f}  ({time.time() - t0:.0f}s)")
        if va_loss < best - 1e-4:
            best, wait = va_loss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE: print("  early stop"); break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist, columns=["train", "val"])

@torch.no_grad()
def dl_predict(model, pairs):
    model.eval()
    return torch.cat([model(get_batch(pairs[b:b + 4096])) for b in range(0, len(pairs), 4096)]).cpu().numpy()

@torch.no_grad()
def dl_loss(model, pairs):
    pred = dl_predict(model, pairs)
    true = np.stack([Y[o:o + H, p] for p, o in pairs])
    return float(np.abs(pred - true).mean())

def dl_forecast(model):
    out = []
    for o in TEST_ORIGINS:
        pairs = np.stack([np.arange(N), np.full(N, o)], axis=1)
        out.append(predict_to_units(dl_predict(model, pairs)).T)          # [H, N]
    return np.stack(out)

## 7. ModernTCN
This uses the **official `ModernTCN` model class** from the cloned repository. Each sample is one product with **4 variables**: sales plus the 3 past covariates, so input `[B, 112, 4]` gives output `[B, 30, 4]`. Only the sales channel is used in the loss. ModernTCN's large depth-wise kernels model each variable over time, and its ConvFFN mixes the 4 variables. RevIN normalises each sample.

The settings follow the repository's ETTh scripts: patch 8 / stride 4, one stage, large kernel 51, small kernel 5, d_model 64, dropout 0.3. The architecture has no input for future calendar or static category information, so those are not used.

In [ ]:
class ModernTCNForecaster(nn.Module):
    def __init__(self):
        super().__init__()
        cfg = SimpleNamespace(
            seq_len=L, pred_len=H, enc_in=1 + COV.shape[-1],
            patch_size=8, patch_stride=4, stem_ratio=6, downsample_ratio=2, ffn_ratio=1,
            num_blocks=[1], large_size=[51], small_size=[5], dims=[64, 64, 64, 64], dw_dims=[64, 64, 64, 64],
            small_kernel_merged=False, dropout=0.3, head_dropout=0.0, use_multi_scale=False,
            revin=1, affine=0, subtract_last=0, decomposition=0, kernel_size=25, individual=0,
            freq="h",   # only used to size an unused time-embedding layer in the official code
        )
        self.net = ModernTCNModel(cfg)

    def forward(self, b):
        x = torch.cat([b["y_hist"].unsqueeze(-1), b["past_cov"]], dim=-1)   # [B, L, 4]
        return self.net(x)[:, :, 0]                                          # sales channel -> [B, H]

torch.manual_seed(SEED)
mtcn, mtcn_hist = fit_dl("ModernTCN", ModernTCNForecaster(), lr=5e-4)
FORECASTS["ModernTCN"] = dl_forecast(mtcn)
summary_table().round(3)

## 8. TiDE
TiDE (*Long-term Forecasting with TiDE: Time-series Dense Encoder*, Das et al., 2023) is implemented here in plain PyTorch:
1. **Feature projection:** a residual MLP compresses the 11 calendar features of each of the 142 days (history + horizon) to 4 dims.
2. **Dense encoder:** concatenates level-normalised sales history, past covariates, projected calendar and **static category embeddings** (brand, Level1–5), then applies 2 residual blocks.
3. **Dense decoder:** 2 residual blocks produce a 16-dim vector for each of the 30 future days.
4. **Temporal decoder:** for each future day, combines that vector with the day's projected calendar (so holidays such as Ashura can act directly).
5. **Linear skip** from the look-back to the horizon.

Sales are normalised per sample by the look-back mean (instance normalisation) and de-normalised at the output.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, d_in, d_hidden, d_out, dropout=0.1, layer_norm=True):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_out), nn.Dropout(dropout))
        self.skip = nn.Linear(d_in, d_out)
        self.norm = nn.LayerNorm(d_out) if layer_norm else nn.Identity()

    def forward(self, x):
        return self.norm(self.net(x) + self.skip(x))


class TiDE(nn.Module):
    def __init__(self, n_past_cov, n_marks, cat_sizes, hidden=256, n_enc=2, n_dec=2, dec_dim=16, proj_dim=4,
                 temporal_hidden=64, emb_dim=8, dropout=0.1):
        super().__init__()
        self.dec_dim = dec_dim
        self.feat_proj = ResBlock(n_marks, 64, proj_dim, dropout)
        self.embs = nn.ModuleList([nn.Embedding(n, emb_dim) for n in cat_sizes])
        enc_in = L + L * n_past_cov + (L + H) * proj_dim + emb_dim * len(cat_sizes)
        self.encoder = nn.Sequential(ResBlock(enc_in, hidden, hidden, dropout),
                                     *[ResBlock(hidden, hidden, hidden, dropout) for _ in range(n_enc - 1)])
        self.decoder = nn.Sequential(*[ResBlock(hidden, hidden, hidden, dropout) for _ in range(n_dec - 1)],
                                     ResBlock(hidden, hidden, H * dec_dim, dropout))
        self.temporal = ResBlock(dec_dim + proj_dim, temporal_hidden, 1, dropout, layer_norm=False)
        self.skip = nn.Linear(L, H)

    def forward(self, b):
        y = b["y_hist"]
        mu = y.mean(dim=1, keepdim=True)                                    # instance normalisation
        y = y - mu
        B = y.shape[0]
        proj = self.feat_proj(b["marks"])                                   # [B, L+H, proj_dim]
        emb = torch.cat([e(b["static"][:, j]) for j, e in enumerate(self.embs)], dim=-1)
        z = torch.cat([y, b["past_cov"].reshape(B, -1), proj.reshape(B, -1), emb], dim=-1)
        dec = self.decoder(self.encoder(z)).reshape(B, H, self.dec_dim)
        out = self.temporal(torch.cat([dec, proj[:, L:]], dim=-1)).squeeze(-1)
        return out + self.skip(y) + mu

torch.manual_seed(SEED)
tide, tide_hist = fit_dl("TiDE", TiDE(n_past_cov=COV.shape[-1], n_marks=MARKS.shape[1], cat_sizes=CAT_SIZES), lr=1e-3)
FORECASTS["TiDE"] = dl_forecast(tide)
summary_table().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.4), sharey=True)
for ax, (name, h_) in zip(axes, [("ModernTCN", mtcn_hist), ("TiDE", tide_hist)]):
    ax.plot(range(1, len(h_) + 1), h_.train, color=PALETTE[0], marker="o", ms=4, label="train")
    ax.plot(range(1, len(h_) + 1), h_.val, color=PALETTE[1], marker="o", ms=4, label="validation")
    ax.set_title(f"{name}: L1 loss (log space)"); ax.set_xlabel("epoch")
axes[0].legend()
plt.tight_layout(); plt.show()

## 9. Comparison

In [ ]:
res = summary_table().sort_values("WAPE")
display(res.style.format({c: "{:.1%}" for c in res.columns if c not in ("MAE", "RMSE")} | {"MAE": "{:,.1f}", "RMSE": "{:,.1f}"}))

base_name = "Baseline (same weekday last week)"
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8), gridspec_kw={"width_ratios": [1, 1.6]})
order = res.index[::-1]
axes[0].barh(order, res.loc[order, "WAPE"] * 100, color=[MODEL_COLORS[m] for m in order], height=0.55)
for y_, v in enumerate(res.loc[order, "WAPE"]):
    axes[0].text(v * 100, y_, f" {v:.1%}", va="center", fontsize=9, color=INK2)
axes[0].set_title("Test WAPE (lower is better)"); axes[0].set_xlabel("%")
for name in FORECASTS:
    axes[1].plot(np.arange(1, H + 1), evaluate(FORECASTS[name])["horizon_WAPE"] * 100, color=MODEL_COLORS[name],
                 lw=2 if name != base_name else 1.5, label=name)
axes[1].set_title("WAPE by forecast day"); axes[1].set_xlabel("days ahead"); axes[1].set_ylabel("%")
axes[1].legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Total sales across all products: actual vs each model, per test window
fig, ax = plt.subplots(figsize=(14, 4))
test_days = np.concatenate([np.arange(o, o + H) for o in TEST_ORIGINS])
ax.plot(dates[test_days], SALES_RAW[test_days].sum(axis=1) / 1e6, color=INK, lw=2, label="actual")
for name, p in FORECASTS.items():
    ax.plot(dates[test_days], p.reshape(-1, N).sum(axis=1) / 1e6, color=MODEL_COLORS[name], lw=1.4, label=name)
for o in TEST_ORIGINS:
    ax.axvline(dates[o], color=AXIS, lw=1)
ax.set_title("Total daily sales in the 3 test windows (millions of units)"); ax.legend(ncol=5, loc="lower center", fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.show()

In [ ]:
# A few individual products (the highest-volume product of 3 different Level1 departments), last test window
fold = N_TEST_FOLDS - 1
o = TEST_ORIGINS[fold]
vol = pd.Series(SALES_RAW[o:o + H].sum(axis=0), index=products)
picks = [vol[meta.index[meta.Level1 == l1]].idxmax() for l1 in sorted(meta.Level1.unique())[:3]]
fig, axes = plt.subplots(len(picks), 1, figsize=(14, 2.6 * len(picks)))
for ax, pid in zip(axes, picks):
    j = products.index(pid)
    hist_idx = np.arange(o - 42, o)
    ax.plot(dates[hist_idx], SALES_RAW[hist_idx, j], color=MUTED, lw=1, label="history")
    ax.plot(dates[o:o + H], SALES_RAW[o:o + H, j], color=INK, lw=2, label="actual")
    for name, p in FORECASTS.items():
        ax.plot(dates[o:o + H], p[fold, :, j], color=MODEL_COLORS[name], lw=1.4, label=name)
    ax.axvline(dates[o], color=AXIS, lw=1)
    ax.set_title(f"{pid} ({meta.loc[pid, 'Level1']})", fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
axes[0].legend(ncol=6, fontsize=8, loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
# Save forecasts (long format) and metrics
rows = []
for name, p in FORECASTS.items():
    for k, o in enumerate(TEST_ORIGINS):
        rows.append(pd.DataFrame({"model": name, "fold": k + 1, "origin": dates[o],
                                  "ds": np.repeat(dates[o:o + H], N), "horizon": np.repeat(np.arange(1, H + 1), N),
                                  "product_id": np.tile(products, H), "forecast": p[k].reshape(-1),
                                  "actual": SALES_RAW[o:o + H].reshape(-1), "inside_life": INSIDE[o:o + H].reshape(-1)}))
fc = pd.concat(rows, ignore_index=True)
suffix = "_smoke" if SMOKE_TEST else ""
fc.to_parquet(OUT_DIR / f"forecasts_test{suffix}.parquet", index=False)
res.to_csv(OUT_DIR / f"model_comparison{suffix}.csv")
print(f"saved {len(fc):,} forecast rows -> {OUT_DIR / f'forecasts_test{suffix}.parquet'}")

best = res.index[0]
b = res.loc[base_name, "WAPE"]
print(f"\nBest model: {best}  |  WAPE {res.loc[best, 'WAPE']:.1%} vs baseline {b:.1%}  "
      f"({res.loc[best, 'skill vs baseline']:+.1%} skill, beats baseline on {res.loc[best, 'product win rate']:.0%} of products)")
for m in res.index:
    if m != base_name:
        print(f"  {m:10s} {'BEATS' if res.loc[m, 'WAPE'] < b else 'does NOT beat'} the baseline "
              f"(WAPE {res.loc[m, 'WAPE']:.1%})")
if SMOKE_TEST:
    print("\nSMOKE_TEST=True: neural models trained for 1 epoch only. Their scores are NOT representative.")

## 10. Notes
- **Fair comparison:** all models see the same 112-day history and the same training period, and are scored on the same three 30-day windows against actual (uncapped) sales.
- **Covariates are past-only.** If the business plans discounts and prices ahead, feed their *future* values too (LightGBM: add target-day discount/price features; TiDE: move them into `marks`). This usually improves promotion periods considerably.
- **Holidays:** `TASUA_ASHURA` covers the data range. Extend it with the official calendar (Eid al-Fitr, Arbaeen, …) before forecasting new periods.
- **Things to try next:**
  - weight the training loss by product volume (aligns training with WAPE);
  - retrain on train + validation before the final forecast;
  - average LightGBM and TiDE, which usually beats either alone;
  - tune `L` (look-back) per model.